# CNN Test Notebook

This notebook contains webcam and single-image inference for the fine-tuned age detector.

Important for Jupyter webcam usage:
1. Do not close the OpenCV window with the mouse.
2. Keep the OpenCV window focused and press `q` to exit safely.
3. Do not run the webcam cell multiple times while the camera is still open.

In [1]:
import importlib.util
import os
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.layers import DepthwiseConv2D
from tensorflow.keras.models import load_model


In [2]:
CONFIG = {
    'model_path': 'runs/cnn_age_detector/best_age_model_finetuned.h5',
    'image_path': 'test_image.png',
    'face_size': (128, 128),
    'camera_index': 0,
    'prediction_threshold': 0.5,
}

CONFIG


{'model_path': 'runs/cnn_age_detector/best_age_model_finetuned.h5',
 'image_path': 'test_image.png',
 'face_size': (128, 128),
 'camera_index': 0,
 'prediction_threshold': 0.5}

In [3]:
def check_and_install():
    print('Checking environment...')
    tf_spec = importlib.util.find_spec('tensorflow')
    if tf_spec is not None:
        print('TensorFlow is already installed. Automatic installation will be skipped.')
        return

    if os.name == 'nt' and sys.version_info >= (3, 11):
        raise RuntimeError('Native Windows TensorFlow GPU support requires Python 3.10 with tensorflow-cpu==2.10 and tensorflow-directml-plugin.')

    print('TensorFlow was not found. Installing a GPU-compatible TensorFlow environment...')
    try:
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            'numpy<2.0',
            'opencv-python==4.8.1.78',
            'tensorflow-cpu==2.10.*',
            'tensorflow-directml-plugin',
            '-q',
        ])
        print('TensorFlow environment installation completed.')
    except Exception as exc:
        print(f'Automatic installation failed: {exc}')


class CustomDepthwiseConv2D(DepthwiseConv2D):
    def __init__(self, **kwargs):
        kwargs.pop('groups', None)
        super().__init__(**kwargs)


def load_age_model(model_path):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Model file not found: {model_path}')

    return load_model(
        model_path,
        custom_objects={'DepthwiseConv2D': CustomDepthwiseConv2D},
    )


def preprocess_face(face_bgr, face_size=(128, 128)):
    face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    face_resized = cv2.resize(face_rgb, face_size)
    face_normalized = face_resized / 255.0
    return np.expand_dims(face_normalized, axis=0)


def predict_face(model, face_bgr, threshold=0.5, face_size=(128, 128)):
    face_input = preprocess_face(face_bgr, face_size=face_size)
    pred_prob = float(model.predict(face_input, verbose=0)[0][0])
    if pred_prob > threshold:
        return f'Adult ({pred_prob * 100:.1f}%)', (0, 0, 255)
    return f'Minor ({(1 - pred_prob) * 100:.1f}%)', (0, 255, 0)


check_and_install()
print(f"Configured model path: {CONFIG['model_path']}")


Checking environment...
TensorFlow is already installed. Automatic installation will be skipped.
Configured model path: runs/cnn_age_detector/best_age_model_finetuned.h5


## Webcam Demo

In [4]:
model = load_age_model(CONFIG['model_path'])
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
cap = cv2.VideoCapture(CONFIG['camera_index'])

if not cap.isOpened():
    print('Error: unable to open the camera.')
else:
    print('Camera started. Press q in the OpenCV window to exit.')

    while True:
        ret, frame = cap.read()
        if not ret:
            print('Failed to read a frame from the camera.')
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

        for (x, y, w, h) in faces:
            face_roi = frame[y:y + h, x:x + w]
            label, color = predict_face(
                model,
                face_roi,
                threshold=CONFIG['prediction_threshold'],
                face_size=CONFIG['face_size'],
            )
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, label, (x, max(y - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        cv2.imshow('AI Age Detector (Press q to exit)', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print('Camera closed safely.')


Camera started. Press q in the OpenCV window to exit.
Camera closed safely.


## Single Image Demo

In [5]:
def analyze_single_image(image_path, model_path=None):
    model_path = model_path or CONFIG['model_path']
    print('Starting single-image analysis...')

    if not os.path.exists(model_path):
        print(f'Error: model file not found: {model_path}')
        return
    if not os.path.exists(image_path):
        print(f'Error: image file not found: {image_path}')
        return

    model = load_age_model(model_path)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    img = cv2.imread(image_path)
    if img is None:
        print('Error: OpenCV could not read the image.')
        return

    img_rgb_display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(50, 50))

    if len(faces) == 0:
        print('No face was detected in this image.')
        plt.figure(figsize=(6, 6))
        plt.imshow(img_rgb_display)
        plt.title('No Face Detected')
        plt.axis('off')
        plt.show()
        return

    print(f'Detected {len(faces)} face(s). Running inference...')

    for (x, y, w, h) in faces:
        face_roi = img[y:y + h, x:x + w]
        label, color_bgr = predict_face(
            model,
            face_roi,
            threshold=CONFIG['prediction_threshold'],
            face_size=CONFIG['face_size'],
        )
        color_rgb = (color_bgr[2], color_bgr[1], color_bgr[0])
        cv2.rectangle(img_rgb_display, (x, y), (x + w, y + h), color_rgb, max(2, int(w * 0.02)))
        cv2.putText(
            img_rgb_display,
            label,
            (x, max(y - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            max(0.5, w * 0.003),
            color_rgb,
            max(1, int(w * 0.01)),
        )

    plt.figure(figsize=(10, 10))
    plt.imshow(img_rgb_display)
    plt.title(f'Prediction Complete: {len(faces)} Face(s) Detected')
    plt.axis('off')
    plt.show()


analyze_single_image(CONFIG['image_path'])


Starting single-image analysis...
Error: image file not found: test_image.png
